In [2]:
!hdfs dfs -mkdir /practica

mkdir: `/practica': File exists


In [5]:
!hdfs dfs -put city_temperature.csv /practica


put: `/practica/city_temperature.csv': File exists


In [4]:
!hdfs dfs -ls /practica


Found 1 items
-rw-r--r--   3 root supergroup  140600832 2025-12-04 06:59 /practica/city_temperature.csv


**Ejercicio 1**

Utiliza MapReduce para encontrar la temperatura máxima registrada para cada ciudad. 

Lógica mapper: lee cada línea de los datos y emite el par (ciudad, temperatura)
 
Lógica reducer: como recibirá todas las líneas de una misma ciudad consecutivamente, emite únicamente la línea con mayor valor en temperatura.

In [ ]:
%%writefile mapper_ejercicio.py
#!/usr/bin/env python3
import os
import sys

for line in sys.stdin:
    line = line.strip()
    region, country, state, city, month, day, year, avg_temperature  = line.split(',')

    print( f"{city}\t{avg_temperature}")

Writing mapper_ejercicio.py


In [ ]:
%%writefile reducer_ejercicio.py
#!/usr/bin/env python3
import sys

ciudad_test = None
temperatura_test = None
for line in sys.stdin:
    city, avg_temperature = line.strip().split("\t",1)
    if ciudad_test is None:
        ciudad_test = city
        temperatura_test = avg_temperature
    if city == ciudad_test:
        if avg_temperature > temperatura_test:
            temperatura_test = avg_temperature
    else:
        print(f"{ciudad_test}\t{temperatura_test}")
        ciudad_test = city
        temperatura_test = avg_temperature
        
if ciudad_test is not None:
    print(f"{ciudad_test}\t{temperatura_test}")

Writing reducer_ejercicio.py


In [ ]:
#Simulamos
!head -n 20 city_temperature.csv | python3 mapper_ejercicio.py | sort | python3 reducer_ejercicio.py

Algiers	64.2
City	AvgTemperature


In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_ejercicio.py \
-file reducer_ejercicio.py \
-mapper mapper_ejercicio.py \
-reducer reducer_ejercicio.py \
-input /practica/city_temperature.csv \
-output /Temperatura_maxima

2025-12-04 07:00:06,734 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_ejercicio.py, reducer_ejercicio.py, /tmp/hadoop-unjar878402213109471649/] [] /tmp/streamjob7432421002524867759.jar tmpDir=null
2025-12-04 07:00:07,957 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.6:8032
2025-12-04 07:00:08,235 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.6:8032
2025-12-04 07:00:08,515 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1764829500936_0010
2025-12-04 07:00:08,996 INFO mapred.FileInputFormat: Total input files to process : 1
2025-12-04 07:00:09,013 INFO net.NetworkTopology: Adding a new node: /default-rack/172.18.0.4:9866
2025-12-04 07:00:09,014 INFO net.NetworkTopology: Adding a new node: /default-rack/172.18.0.2:9866
2025-12-04 07:00:09,015

In [ ]:
!hdfs dfs -ls /

In [ ]:
!hdfs dfs -ls /Temperatura_maxima


In [ ]:
!hdfs dfs -cat /Temperatura_maxima/part-00000


**Ejercicio 2**

Calcula la temperatura media histórica para cada país. El proceso es similar al anterior, en el reducer debes ir recordando todos los datos que te lleguen de un mismo país y, cuando pase al siguiente país, calcular la media y emitirlos.

In [ ]:
%%writefile mapper_ejercicio2.py
#!/usr/bin/env python3
import os
import sys

for line in sys.stdin:
    line = line.strip()
    region, country, state, city, month, day, year, avg_temperature  = line.split(',')

    print( f"{country}\t{avg_temperature}")

In [ ]:
%%writefile reducer_ejercicio2.py
#!/usr/bin/env python3

import sys
country_test = None
temperatura_test = 0
contador = 0
for line in sys.stdin:
    country, avg_temperature = line.strip().split("\t",1)
    if avg_temperature == "AvgTemperature":
        continue
    else:
        avg_temperature = float(avg_temperature)

    if country_test is None:
        country_test = country
    if country == country_test:
        contador += 1
        temperatura_test = temperatura_test + avg_temperature
    else:
        temperatura_test = temperatura_test / contador
        print(f"{country_test}\t{temperatura_test}")
        country_test = country
        temperatura_test = avg_temperature
        contador =  1
        
if country_test is not None:
    temperatura_test = temperatura_test / contador
    print(f"{country_test}\t{temperatura_test}")


In [ ]:
#Simulamos
!head -n 30000 city_temperature.csv | python3 mapper_ejercicio2.py | sort | python3 reducer_ejercicio2.py

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_ejercicio2.py \
-file reducer_ejercicio2.py \
-mapper mapper_ejercicio2.py \
-reducer reducer_ejercicio2.py \
-input /practica/city_temperature.csv \
-output /Temperatura_media_pais

In [ ]:
!hdfs dfs -ls /Temperatura_media_pais

In [ ]:
!hdfs dfs -cat /Temperatura_media_pais/part-00000

**Ejercicio 3**

Calcula cuántos días calurosos (definidos como >30 grados) hubo en cada año en cada ciudad.

In [ ]:
%%writefile mapper_ejercicio3.py
#!/usr/bin/env python3
import os
import sys

for line in sys.stdin:
    line = line.strip()
    region, country, state, city, month, day, year, avg_temperature  = line.split(',')

    print( f"{city}\t{avg_temperature}")

In [ ]:
%%writefile reducer_ejercicio3.py
#!/usr/bin/env python3
import sys

city_test = None
contador = 0
for line in sys.stdin:
    city, avg_temperature = line.strip().split("\t",1)
    if avg_temperature == "AvgTemperature":
        continue
    else:
        avg_temperature = float(avg_temperature)
    if city_test is None:
       city_test = city
    if city == city_test:
        if avg_temperature > 30:
            contador += 1

    else:
        print(f"{city_test}\t{contador}")
        city_test = city
        contador =  0
        
if city_test is not None:
    print(f"{city_test}\t{contador}")

In [ ]:
#Simulamos
!head -n 30000 city_temperature.csv | python3 mapper_ejercicio3.py | sort | python3 reducer_ejercicio3.py

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_ejercicio3.py \
-file reducer_ejercicio3.py \
-mapper mapper_ejercicio3.py \
-reducer reducer_ejercicio3.py \
-input /practica/city_temperature.csv \
-output /Temperatura_30_ciudad

In [ ]:
!hdfs dfs -ls /Temperatura_30_ciudad

In [ ]:
!hdfs dfs -cat /Temperatura_30_ciudad/part-00000


**Ejercicio 4**

Encuentra la temperatura mínima y máxima registrada para cada región.

In [ ]:
%%writefile mapper_ejercicio4.py
#!/usr/bin/env python3
import os
import sys

for line in sys.stdin:
    line = line.strip()
    region, country, state, city, month, day, year, avg_temperature  = line.split(',')

    print( f"{city}\t{avg_temperature}")

In [ ]:
%%writefile reducer_ejercicio4.py
#!/usr/bin/env python3
import sys

ciudad_test = None
temperatura_testMax = None
temperatura_testMin = None
for line in sys.stdin:
    city, avg_temperature = line.strip().split("\t",1)
    if avg_temperature == "AvgTemperature":
        continue
    else:
        avg_temperature = float(avg_temperature)
    if avg_temperature == -99.0: 
        continue
    if ciudad_test is None:
        ciudad_test = city
        temperatura_testMax = avg_temperature
        temperatura_testMin = avg_temperature
    if city == ciudad_test:
        if avg_temperature > temperatura_testMax:
            temperatura_testMax = avg_temperature
        if avg_temperature < temperatura_testMin:
            temperatura_testMin = avg_temperature
    else:
        print(f"{ciudad_test}\t{temperatura_testMax}\t {temperatura_testMin}")
        ciudad_test = city
        temperatura_testMax = avg_temperature
        temperatura_testMin = avg_temperature
        
if ciudad_test is not None:
    print(f"{ciudad_test}\t{temperatura_testMax}\t {temperatura_testMin}")

In [ ]:
#Simulamos
!head -n 30000 city_temperature.csv | python3 mapper_ejercicio4.py | sort | python3 reducer_ejercicio4.py

In [ ]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_ejercicio4.py \
-file reducer_ejercicio4.py \
-mapper mapper_ejercicio4.py \
-reducer reducer_ejercicio4.py \
-input /practica/city_temperature.csv \
-output /Temperatura_MinMax

In [ ]:
!hdfs dfs -ls /Temperatura_MinMax

In [ ]:
!hdfs dfs -cat /Temperatura_MinMax/part-00000
